# Wan 2.1 T2V 1.3B — Tesla T4 + 12 GB RAM low-memory notebook

**Run cells from top to bottom.** Start with a small test clip. This notebook deliberately uses a smaller 384×640 test resolution and 33 frames (~2 seconds) to reduce memory pressure.

Important:
- This is a best-effort low-memory setup, not a guarantee. Wan 2.1 still has substantial memory requirements.
- Do not run other large notebooks in the same runtime.
- If loading still exhausts 12 GB RAM, use a higher-RAM Colab runtime or a hosted GPU with more system RAM.
- Once the test works, increase resolution/frames cautiously. A 1–2 minute movie requires many clips and substantial compute time.

In [ ]:
# 1) Install compatible packages
!pip -q install -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg

In [ ]:
# 2) Restart runtime after installation if Colab asks, then run this cell
import os, gc, time, subprocess, torch
from pathlib import Path

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "GPU runtime required: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

# Avoid excessive CPU thread use on small RAM runtimes
torch.set_num_threads(2)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!free -h

In [ ]:
# 3) Load model with CPU offload
# Important: use float16 on Tesla T4. Load VAE in float32 for numerical stability,
# then let Diffusers CPU offload manage module placement.
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

gc.collect()
torch.cuda.empty_cache()

vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config,
    flow_shift=3.0
)

# Sequential CPU offload is more memory-conservative than keeping all modules on GPU.
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()

print("Model loaded with sequential CPU offload.")
!free -h

In [ ]:
# 4) Low-memory test generation — about 2 seconds
import uuid, gc, torch, os
from pathlib import Path

OUT = Path("/content/wan_t4_outputs")
OUT.mkdir(exist_ok=True)

def generate_test(prompt, steps=15, seed=42):
    prompt = (prompt or "").strip()
    if len(prompt) < 8:
        raise ValueError("Please enter a more detailed prompt.")

    gc.collect()
    torch.cuda.empty_cache()

    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    result = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality, flicker, jitter, distorted anatomy, text, watermark",
        height=384,
        width=640,
        num_frames=33,
        num_inference_steps=int(steps),
        guidance_scale=4.5,
        generator=generator
    )
    frames = result.frames[0]
    path = OUT / f"test_{uuid.uuid4().hex[:8]}.mp4"
    export_to_video(frames, str(path), fps=16)

    del result, frames, generator
    gc.collect()
    torch.cuda.empty_cache()
    return str(path)

print("Test function ready. Run the next cell.")

In [ ]:
# 5) Generate one test clip
test_prompt = "A cinematic wide shot of a small cabin in a snowy pine forest at sunrise, gentle snow falling, warm light glowing from windows, slow camera push in, realistic film look"
test_video = generate_test(test_prompt, steps=15, seed=42)
print("Generated:", test_video)

In [ ]:
# 6) Preview and download test video
from IPython.display import Video, display
display(Video(test_video, embed=True))
from google.colab import files
files.download(test_video)

## If the test succeeds: generate a longer project cautiously

For a 60–120 second video, generate one short clip at a time and save it immediately. A 2-second clip means about 30 clips for one minute or 60 clips for two minutes. This is compute-intensive and may exceed free Colab session limits.

Do not jump straight to 1–2 minutes. First confirm the test cell runs without a RAM crash.

In [ ]:
# 7) Optional: inspect memory before attempting longer generation
!free -h
!nvidia-smi